# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes without subscripting
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Dataset Identifier: {dataset.metadata.identifier}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note**: All entities are referenced using their `@id` field, which uniquely identifies each record set or field in the Croissant schema.

In [ ]:
# List available RecordSets and their IDs
print("Available Record Sets (@id):")
for record_set in dataset.metadata.recordSets:
    print(f"- name: {record_set.name} | @id: {record_set.id}")

# For demonstration, enumerate fields for the first RecordSet
record_sets = dataset.metadata.recordSets
if record_sets:
    first_rs = record_sets[0]
    print(f"\nFields for RecordSet '{first_rs.name}' (@id: {first_rs.id}):")
    for field in first_rs.fields:
        print(f"  - {field.name} (@id: {field.id}) | dataType: {field.dataType}")
else:
    print("No record sets found in this dataset. Please review the Croissant schema.")

## 3. Data Extraction
Load data from all record sets into DataFrames. The `@id` of each record set and their fields are referenced as defined above.

In [ ]:
# Extract data from all available RecordSets by their @id
dataframes = {}

# Collect all record_set @ids
record_sets = dataset.metadata.recordSets
record_set_ids = [rs.id for rs in record_sets]
print(f"Found {len(record_set_ids)} record sets.")

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded RecordSet: {record_set_id} – {len(df)} records, columns: {df.columns.tolist()}")

# Example: print the first 5 rows for the main RecordSet
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in main record set ({main_record_set_id}):\n", dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA steps:
- Filtering records by a numeric field
- Normalizing numeric columns
- Grouping by categorical variables

We reference all field/column names and record set identifiers by their `@id` value throughout.

In [ ]:
# Choose main RecordSet for analysis
if record_set_ids:
    record_set_id = main_record_set_id
    df = dataframes[record_set_id]
    print(f"\nPreview data from RecordSet {record_set_id}:")
    display(df.head())

    # Identify a numeric field by inspecting the DataFrame's dtypes
    print("\nNumeric columns:")
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    print(numeric_cols)
    
    if numeric_cols:
        numeric_field = numeric_cols[0]  # Example: pick the first numeric column
        # Filtering
        threshold = df[numeric_field].mean()  # Use the mean as a threshold example
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"\nFiltered records with {numeric_field} > {threshold:.2f} (showing up to 5):")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() != 0 else 1)
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field if available
        potential_groupers = df.select_dtypes(include=['object']).columns.tolist()
        if potential_groupers:
            group_field = potential_groupers[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field}, averaged {numeric_field}:")
            display(grouped_df.head())
        else:
            print("No categorical fields available for grouping.")
    else:
        print("This record set does not contain any numeric fields for analysis.")

## 5. Visualization
Visualize the distribution of the primary numeric field and its relationship to any available categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if record_set_ids and numeric_cols:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    plt.title(f"Distribution of {numeric_field} in {record_set_id}")
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group_field if available
    if potential_groupers:
        plt.figure(figsize=(10, 5))
        plt.title(f"{numeric_field} distribution by {group_field}")
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² dataset on human protein mass spectrometry using the `mlcroissant` library. We referenced all data entities by their unique `@id` as defined in the Croissant schema, listed available record sets and fields, loaded data into DataFrames, applied basic EDA techniques, and visualized field distributions.

- For further analysis, consult the Croissant metadata or documentation for detailed schema, column meanings, and licensing requirements.
- All dataset elements can be programmatically accessed through the `mlcroissant` interface using their respective `@id`s for reproducible and robust workflows.